# Data prep in SQL (DuckDB)

This notebook rebuilds the data prep from notebooks 01 to 03 using SQL queries in DuckDB, then checks that the results match the pandas outputs saved in `data/processed/`. It only reads files; nothing in `data/processed/` is overwritten.

## Setup

In [1]:
import os

import duckdb
import pandas as pd

con = duckdb.connect()

GRAD_RAW = '../data/raw/SY1018_FS150_FS151_DG695_DG696_LEA.csv'
CHRONIC_RAW = '../data/raw/ID 814 SCH - Chronic Absenteeism.csv'
ENROLL_RAW = '../data/raw/Enrollment.csv'

GRAD_PANDAS = '../data/processed/edfacts_lea_graduation_2017_2018_v2.csv'
ABS_PANDAS = '../data/processed/crdc_lea_absenteeism_2017_18_v2.csv'
MERGED_PANDAS = '../data/processed/lea_absenteeism_x_graduation_v2.csv'

# The raw EDFacts file is too large for GitHub, so it may not be present
GRAD_FROM_RAW = os.path.exists(GRAD_RAW)
print('Raw EDFacts file found:', GRAD_FROM_RAW)

Raw EDFacts file found: False


## Load raw files

Load every column as text so each conversion to a number is an explicit step, matching `pd.to_numeric(errors='coerce')` in the pandas notebooks.

In [2]:
if GRAD_FROM_RAW:
    con.sql(f"CREATE TABLE edfacts_raw AS SELECT * FROM read_csv('{GRAD_RAW}', all_varchar = true)")

con.sql(f"CREATE TABLE chronic_raw AS SELECT * FROM read_csv('{CHRONIC_RAW}', all_varchar = true)")
con.sql(f"CREATE TABLE enrollment_raw AS SELECT * FROM read_csv('{ENROLL_RAW}', all_varchar = true)")

con.sql('''
    SELECT 'chronic_raw' AS table_name, COUNT(*) AS n_rows FROM chronic_raw
    UNION ALL
    SELECT 'enrollment_raw', COUNT(*) FROM enrollment_raw
''').df()

,table_name,n_rows
0,chronic_raw,92437
1,enrollment_raw,97632


## Graduation rates (notebook 01)

Keep 2017-18 rows for all students in cohorts of at least 100, and fix the two range codes Excel turned into dates.

In [3]:
if GRAD_FROM_RAW:
    con.sql('''
        CREATE VIEW grad_filtered AS
        SELECT
            CAST("NCES LEA ID" AS BIGINT) AS LEAID,
            CASE "Value"
                WHEN '14-Oct' THEN '10-14'
                WHEN '9-Jun' THEN '6-9'
                ELSE "Value"
            END AS value_code
        FROM edfacts_raw
        WHERE "Subgroup" = 'All Students'
          AND "School Year" = '2017-2018'
          AND TRY_CAST("Denominator" AS DOUBLE) >= 100
    ''')

Turn each suppressed code into one number: GE/LE codes use the boundary, ranges use the midpoint, and keep only rates between 0 and 100.

In [4]:
if GRAD_FROM_RAW:
    grad_sql = con.sql('''
        SELECT DISTINCT LEAID, grad_rate
        FROM (
            SELECT
                LEAID,
                CASE
                    WHEN value_code LIKE 'GE%' THEN TRY_CAST(SUBSTR(value_code, 3) AS DOUBLE)
                    WHEN value_code LIKE 'LE%' THEN TRY_CAST(SUBSTR(value_code, 3) AS DOUBLE)
                    WHEN value_code LIKE '%-%' THEN
                        (TRY_CAST(SPLIT_PART(value_code, '-', 1) AS DOUBLE)
                         + TRY_CAST(SPLIT_PART(value_code, '-', 2) AS DOUBLE)) / 2
                    ELSE TRY_CAST(value_code AS DOUBLE)
                END AS grad_rate
            FROM grad_filtered
        )
        WHERE grad_rate BETWEEN 0 AND 100
        ORDER BY LEAID
    ''').df()
else:
    # Without the raw file, the join below uses the saved pandas output for graduation
    grad_sql = pd.read_csv(GRAD_PANDAS)

con.register('grad_lea', grad_sql)
print('Graduation LEAs:', len(grad_sql))

Graduation LEAs: 6113


## Chronic absenteeism (notebook 02)

Convert the chronic absence counts to numbers and treat negative CRDC reserve codes as missing.

In [5]:
con.sql('''
    CREATE VIEW chronic_clean AS
    SELECT
        CAST(NCESLEAID AS BIGINT) AS LEAID,
        CAST(NCESSCH AS BIGINT) AS NCESSCH,
        CASE WHEN TRY_CAST(TOTAL_STUDENTS_REPORTED_M AS DOUBLE) >= 0
             THEN TRY_CAST(TOTAL_STUDENTS_REPORTED_M AS DOUBLE) END AS absent_m,
        CASE WHEN TRY_CAST(TOTAL_STUDENTS_REPORTED_F AS DOUBLE) >= 0
             THEN TRY_CAST(TOTAL_STUDENTS_REPORTED_F AS DOUBLE) END AS absent_f
    FROM chronic_raw
''')

Do the same for K-12 enrollment; the 510 alphanumeric California `COMBOKEY` values become NULL and drop out at the join.

In [6]:
con.sql('''
    CREATE VIEW enrollment_clean AS
    SELECT
        TRY_CAST(COMBOKEY AS BIGINT) AS COMBOKEY,
        CASE WHEN TRY_CAST(TOT_ENR_M AS DOUBLE) >= 0
             THEN TRY_CAST(TOT_ENR_M AS DOUBLE) END AS enroll_m,
        CASE WHEN TRY_CAST(TOT_ENR_F AS DOUBLE) >= 0
             THEN TRY_CAST(TOT_ENR_F AS DOUBLE) END AS enroll_f
    FROM enrollment_raw
''')

con.sql('SELECT COUNT(*) AS non_numeric_combokey FROM enrollment_clean WHERE COMBOKEY IS NULL').df()

,non_numeric_combokey
0,510


Join the two files at the school level and add male and female counts, returning NULL only when both are missing.

In [7]:
# The CASE mirrors pandas sum(axis=1, min_count=1)
con.sql('''
    CREATE VIEW school_counts AS
    SELECT
        c.LEAID,
        c.NCESSCH,
        CASE WHEN c.absent_m IS NULL AND c.absent_f IS NULL THEN NULL
             ELSE COALESCE(c.absent_m, 0) + COALESCE(c.absent_f, 0) END AS chron_absent,
        CASE WHEN e.enroll_m IS NULL AND e.enroll_f IS NULL THEN NULL
             ELSE COALESCE(e.enroll_m, 0) + COALESCE(e.enroll_f, 0) END AS enrolled
    FROM chronic_clean AS c
    INNER JOIN enrollment_clean AS e
        ON c.NCESSCH = e.COMBOKEY
''')

con.sql('''
    SELECT
        COUNT(*) AS schools_joined,
        COUNT(*) - COUNT(DISTINCT NCESSCH) AS duplicate_schools,
        COUNT(*) FILTER (WHERE chron_absent IS NOT NULL AND enrolled IS NULL) AS absent_known_enrolled_missing
    FROM school_counts
''').df()

,schools_joined,duplicate_schools,absent_known_enrolled_missing
0,89224,0,10


Drop schools with no enrollment, then add up counts by district and compute the weighted absence rate.

In [8]:
con.sql('''
    CREATE VIEW lea_all AS
    SELECT
        LEAID,
        SUM(chron_absent) AS chron_absent,
        SUM(enrolled) AS enrolled,
        SUM(chron_absent) / SUM(enrolled) * 100 AS absent_rate
    FROM school_counts
    WHERE enrolled IS NOT NULL
    GROUP BY LEAID
''')

con.sql('''
    SELECT
        COUNT(*) AS districts,
        COUNT(*) FILTER (WHERE absent_rate > 100) AS over_100_pct,
        COUNT(*) FILTER (WHERE absent_rate IS NULL) AS null_rate
    FROM lea_all
''').df()

,districts,over_100_pct,null_rate
0,15998,188,0


Exclude the districts whose rate is above 100%, which are mostly technical centers and high-mobility charter networks.

In [9]:
abs_sql = con.sql('''
    SELECT LEAID, chron_absent, enrolled, absent_rate
    FROM lea_all
    WHERE absent_rate <= 100
    ORDER BY LEAID
''').df()

con.register('abs_lea', abs_sql)
print('Absenteeism LEAs:', len(abs_sql))

Absenteeism LEAs: 15810


## Join graduation and absenteeism (notebook 03)

Keep only districts that have both a graduation rate and an absence rate.

In [10]:
merged_sql = con.sql('''
    SELECT g.LEAID, g.grad_rate, a.chron_absent, a.enrolled, a.absent_rate
    FROM grad_lea AS g
    INNER JOIN abs_lea AS a
        ON g.LEAID = a.LEAID
    ORDER BY g.LEAID
''').df()

print('Joined LEAs:', len(merged_sql))

Joined LEAs: 5940


## Check SQL against pandas

Compare each SQL result to the matching pandas output: row counts, district IDs, then every value with `pd.testing.assert_frame_equal`.

In [11]:
def compare(name, sql_df, pandas_path):
    pandas_df = pd.read_csv(pandas_path)

    # Row order and integer widths differ between the tools, so sort and align types before comparing
    sql_df = sql_df.sort_values('LEAID').reset_index(drop=True)
    pandas_df = pandas_df.sort_values('LEAID').reset_index(drop=True)
    sql_df = sql_df.astype(pandas_df.dtypes.to_dict())

    print(f'{name}: SQL rows = {len(sql_df)}, pandas rows = {len(pandas_df)}')
    assert len(sql_df) == len(pandas_df), f'{name}: row counts differ'

    only_sql = set(sql_df['LEAID']) - set(pandas_df['LEAID'])
    only_pandas = set(pandas_df['LEAID']) - set(sql_df['LEAID'])
    assert not only_sql and not only_pandas, (
        f'{name}: LEAIDs differ. Only in SQL: {sorted(only_sql)[:10]}, '
        f'only in pandas: {sorted(only_pandas)[:10]}'
    )

    pd.testing.assert_frame_equal(sql_df, pandas_df, check_exact=False, rtol=1e-9)
    print(f'{name}: match\n')


if GRAD_FROM_RAW:
    compare('Graduation', grad_sql, GRAD_PANDAS)
else:
    print('Graduation: skipped, raw EDFacts file not in data/raw/\n')

compare('Absenteeism', abs_sql, ABS_PANDAS)
compare('Joined', merged_sql, MERGED_PANDAS)

Graduation: skipped, raw EDFacts file not in data/raw/

Absenteeism: SQL rows = 15810, pandas rows = 15810
Absenteeism: match

Joined: SQL rows = 5940, pandas rows = 5940
Joined: match

